# 실무 지식 그래프(Knowledge Graph) 응용

**코스 모듈:** Module 8
**대상:** Neo4j, Cypher 기초 및 (선택) LLM을 이해하는 초급 학습자

## 코스 설명

지식 그래프(Knowledge Graph)는 단순한 저장 형식이 아닙니다. 관계가 의사결정을 이끄는 **실제 애플리케이션**을 구동합니다:
물류, 규정 준수(compliance), 리스크, 추천, 근거 기반 질의응답(question answering) 등입니다.

이 실습 코스에서는 다음을 수행합니다:

1. Neo4j에서 **해운/공급망(supply-chain) 지식 그래프**를 구축합니다.
2. Cypher로 **7가지 실무 문제**를 해결합니다(LLM 보조 Q&A 포함).
3. 각 연습을 업계에서 들을 수 있는 **비즈니스 질문**과 연결합니다.

> **언어:** 이 노트북의 안내 텍스트는 **한국어**입니다. 기술 용어는 필요 시 영어를 병기합니다.

> **설정(필수):** 코드 실행 전 **`NEO4J_SETUP.md`**를 완료하세요.
> LLM 섹션의 경우 **`LLM_MODEL_SETUP.md`**를 완료하고 **`ollama_model_runner.py`**를 실행하세요
> (또는 OpenAI를 구성).

### 이 노트북 사용법

1. 아래 **코드** 셀을 실행하기 전에 각 **마크다운** 셀을 읽으세요.
2. Step 0부터 **순서대로** 셀을 실행하세요.
3. 랩 데이터는 **`KGApplicationsLab`** 레이블을 사용합니다 — 안전하게 삭제 후 재시드할 수 있습니다.
4. 코드 셀은 의도적으로 **짧게** 작성되었습니다. 마크다운이 각 단계의 *이유*를 설명합니다.


## 사전 요구 사항

| 기술 | 필요한 이유 |
|-------|-----------------|
| Neo4j 기초 | 노드, 관계, `MATCH`, `RETURN` |
| Python | 노트북 셀 실행 |
| (선택) LLM | Part 8 — 자연어 비즈니스 Q&A |

### 관련 Module 8 노트북

| 노트북 | 초점 |
|----------|-------|
| `Module_8_Building_Knowledge_Graphs_with_LLMs.ipynb` | LLM으로 텍스트에서 그래프 추출 |
| `Module_8_Using_Knowledge_Graph_with_LangChain.ipynb` | LangChain RAG / GraphRAG / agent |
| `Module_8_Evaluating_GraphRAG_with_RAGAS.ipynb` | GraphRAG 품질 측정 |

**이 노트북**은 그래프가 존재한 **이후에 할 수 있는 일**에 초점을 맞춥니다 — 프레임워크 API가 아닌 **응용(application)**입니다.

## 코스 개요

| 파트 | 주제 | 실무 문제 |
|------|--------|-------------------|
| **0** | 환경 및 Neo4j 연결 | — |
| **1** | 응용 지형 | 그래프가 테이블보다 나은 경우 |
| **2** | 랩 그래프 구축 | 해운 물류 도메인 모델 |
| **3** | 네트워크 탐색 | *누가 어디서 운영하나?* |
| **4** | 경로 및 영향 분석 | *경로가 중단되면 무엇이 깨지나?* |
| **5** | 엔티티 해소(entity resolution) | *이 두 레코드가 같은 회사인가?* |
| **6** | 추천 | *어떤 항구가 유사한가?* |
| **7** | 규정 준수 매핑 | *어떤 규칙이 어떤 운송사에 적용되나?* |
| **8** | 비즈니스 Q&A (LLM + Cypher) | *일반 영어로 질문하기* |
| **9** | 그래프 분석 | *어떤 허브가 가장 중요한가?* |
| **10** | 마무리 | 연습을 자신의 도메인에 매핑 |


---

# Step 0 — 환경 및 Neo4j 연결

### 코드 실행 전

1. **`NEO4J_SETUP.md`** 완료 — Aura, Desktop 또는 Docker; URI, 사용자명, 비밀번호 확인.
2. `Module_8/.env.example` → `Module_8/.env` 복사 후 Neo4j 자격 증명 입력.
3. **Part 8**(LLM Q&A)의 경우 **`LLM_MODEL_SETUP.md`**도 완료하고 `ollama serve`를 시작하세요.

### Step 0에서 하는 일

| Step | 목적 |
|------|---------|
| 0a | Python 패키지 설치 |
| 0b | 경로 및 환경 변수 로드 |
| 0c | Neo4j Bolt 연결 확인 |
| 0d | (선택) Part 8용 Ollama runner 연결 |


### Step 0a — Python 패키지 설치

핵심 패키지: Neo4j driver, `python-dotenv`, Part 8에서 사용하는 LangChain 구성 요소.
가상 환경당 한 번 실행하세요.


In [1]:
# Step 0a — Install Python dependencies (run once per environment)
%pip install -q neo4j python-dotenv requests \
    langchain langchain-community langchain-neo4j langchain-openai


Note: you may need to restart the kernel to use updated packages.


### Step 0b.1 — `Module_8` 폴더 확인

Jupyter는 저장소 루트 또는 `Module_8` 내부에서 시작할 수 있습니다. 이 셀은 `.env`와 `data/`가 있는 폴더를 찾습니다.


In [2]:
# Step 0b.1 — Resolve Module_8 directory
import os
from pathlib import Path
from dotenv import load_dotenv

MODULE_DIR = Path('.').resolve()
if MODULE_DIR.name != 'Module_8':
    _candidate = MODULE_DIR / 'Module_8'
    if _candidate.is_dir():
        MODULE_DIR = _candidate.resolve()
load_dotenv(MODULE_DIR / '.env')
print('모듈 디렉터리:', MODULE_DIR)


모듈 디렉터리: /home/ethan/newgen/KMOU_Course/Module_8


### Step 0b.2 — Neo4j 연결 설정

값은 `Module_8/.env`에서 가져옵니다(**`NEO4J_SETUP.md`** 참고).


In [3]:
# Step 0b.2 — Neo4j settings
NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7687')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')
print('Neo4j URI:', NEO4J_URI)
print('Neo4j 데이터베이스:', NEO4J_DATABASE)


Neo4j URI: neo4j://localhost:7687
Neo4j 데이터베이스: neo4j


### Step 0b.3 — LLM 설정 (Part 8 전용)

Part 1–7은 **Cypher만** 사용합니다. Part 8에서 자연어 Q&A를 추가합니다.

- **`LLM_BACKEND=ollama`** → `ollama_model_runner.py` (권장)
- **`LLM_BACKEND=openai`** → `ChatOpenAI`


In [4]:
# Step 0b.3 — LLM backend (used in Part 8)
LLM_BACKEND = os.getenv('LLM_BACKEND', 'ollama').lower()
OLLAMA_HOST = os.getenv('OLLAMA_HOST', 'http://localhost:11434')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'llama3.2:3b')
OLLAMA_TEMPERATURE = float(os.getenv('OLLAMA_TEMPERATURE', '0'))
OLLAMA_MAX_TOKENS = int(os.getenv('OLLAMA_MAX_TOKENS', '2048'))
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
LAB_LABEL = 'KGApplicationsLab'
print('LLM 백엔드:', LLM_BACKEND)
if LLM_BACKEND == 'ollama':
    print('Ollama 호스트:', OLLAMA_HOST)
    print('Ollama 모델:', OLLAMA_MODEL)


LLM 백엔드: ollama
Ollama 호스트: http://localhost:11434
Ollama 모델: llama3.1:8b


### Step 0c — Neo4j 연결 확인

실패하면 **`NEO4J_SETUP.md`** 문제 해결 섹션으로 돌아가세요.


In [5]:
# Step 0c — Verify Neo4j connectivity
from neo4j import GraphDatabase

if not NEO4J_PASSWORD:
    raise ValueError(
        'NEO4J_PASSWORD is empty. Set it in Module_8/.env (see NEO4J_SETUP.md).'
    )

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run('RETURN "Neo4j connection OK" AS message').single()
    print(record['message'])
driver.close()
print('연결 확인 통과.')


Neo4j connection OK
연결 확인 통과.


### Step 0d — Neo4j 헬퍼 및 (선택) Ollama runner

노트북 전체에서 사용하는 작은 `run_cypher()` 헬퍼를 정의합니다.
Ollama runner 블록은 **Part 8**에만 필요합니다 — 원하면 0d.2–0d.4는 나중에 건너뛸 수 있습니다.


In [6]:
# Step 0d.1 — Cypher helper
def run_cypher(query: str, params: dict | None = None) -> list[dict]:
    """Run a read/write Cypher query and return rows as dicts."""
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as drv:
        with drv.session(database=NEO4J_DATABASE) as session:
            result = session.run(query, params or {})
            return [dict(r) for r in result]

print('run_cypher() 준비 완료.')


run_cypher() 준비 완료.


In [7]:
# Step 0d.2 — Resolve ollama_model_runner.py (Part 8)
import json
import subprocess
import sys
import tempfile
from typing import Any, List, Optional

from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM


def _resolve_ollama_runner_path() -> Path:
    for candidate in (
        MODULE_DIR / 'ollama_model_runner.py',
        MODULE_DIR.parent / 'Module_4' / 'ollama_model_runner.py',
        Path('ollama_model_runner.py'),
    ):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'ollama_model_runner.py not found. See LLM_MODEL_SETUP.md.'
    )


OLLAMA_RUNNER = _resolve_ollama_runner_path()
print('OLLAMA_RUNNER:', OLLAMA_RUNNER)


OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_8/ollama_model_runner.py


In [8]:
# Step 0d.3 — call_ollama_runner() helper
def call_ollama_runner(prompt: str, *, model: str | None = None) -> str:
    model_arg = model or OLLAMA_MODEL
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as pf:
        path = pf.name
        pf.write(prompt)
    try:
        cmd = [
            sys.executable, str(OLLAMA_RUNNER),
            '--host', OLLAMA_HOST,
            '--models', model_arg,
            '--prompt-file', path,
            '--temperature', str(OLLAMA_TEMPERATURE),
            '--max-tokens', str(OLLAMA_MAX_TOKENS),
        ]
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode != 0:
            raise RuntimeError(f'Runner exit {run.returncode}\n{run.stderr[:2000]}')
        payload = json.loads(run.stdout)
        first = (payload.get('outputs') or [{}])[0]
        if first.get('status') != 'ok':
            raise RuntimeError(f'Ollama error: {first}')
        return (first.get('response') or '').strip()
    finally:
        Path(path).unlink(missing_ok=True)


class OllamaRunnerLLM(LLM):
    model: str = OLLAMA_MODEL

    @property
    def _llm_type(self) -> str:
        return 'ollama_runner'

    def _call(
        self, prompt: str, stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None, **kwargs: Any,
    ) -> str:
        return call_ollama_runner(prompt, model=self.model)

print('Ollama 헬퍼 정의 완료.')


Ollama 헬퍼 정의 완료.


In [9]:
# Step 0d.4 — Instantiate llm for Part 8
if LLM_BACKEND == 'openai':
    if not os.getenv('OPENAI_API_KEY'):
        raise ValueError('OPENAI_API_KEY required when LLM_BACKEND=openai')
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    print(f'LLM 준비 완료: OpenAI {OPENAI_MODEL}')
elif LLM_BACKEND == 'ollama':
    llm = OllamaRunnerLLM()
    print(f'LLM 준비 완료: Ollama {OLLAMA_MODEL} (runner 경유)')
else:
    llm = None
    print('LLM 미구성 — LLM_BACKEND를 설정하지 않으면 Part 8을 건너뜁니다.')


LLM 준비 완료: Ollama llama3.1:8b (runner 경유)


---

# Part 1 — 실무 지식 그래프 응용

관계형 데이터베이스는 **행과 조인**에 강합니다. 지식 그래프는 다음 경우에 강합니다:

- **연결 구조** 자체가 제품인 경우(공급망, 조직도, 사기 네트워크).
- **다중 홉(multi-hop) 추론**이 필요한 경우(*공급업체 → 공장 → 항구 → 운하*).
- 여러 팀이 **단일 연결 모델**을 공유하는 경우(운영, 규정 준수, 데이터 과학).

### 응용 지도 (이 코스)

| 산업 | 그래프 응용 | 노트북 파트 |
|----------|-------------------|---------------|
| 물류 | 경로 및 허브 분석 | 3, 4, 9 |
| 소매 | 제품 추천 | 6 |
| 금융 | 엔티티 해소, AML 경로 | 5 |
| 헬스케어 | 치료 경로 | (Part 4와 동일한 패턴) |
| 기업 | 정책 / 규정 준수 매핑 | 7 |
| GenAI | 근거 기반 비즈니스 Q&A | 8 |

### 실습 도메인: 해운 및 공급망

**항구, 운송사(carrier), 운하, 국가, 규정**을 사용합니다 — 글로벌 무역의 현실적인 단면입니다.
데이터는 Cypher(구조화)로 시드되며, 선택적으로 `data/dbpedia_maritime_corpus.txt`의 텍스트를 사용합니다(`data/DATASETS.md` 참고).


---

# Part 2 — 응용 지식 그래프 구축

## 2.1 데이터 모델

```text
(Port)-[:LOCATED_IN]->(Country)
(Organization)-[:OPERATES_IN]->(Port)
(Organization)-[:USES_ROUTE]->(Waterway)
(Organization)-[:HEADQUARTERED_IN]->(Country)
(Regulation)-[:ISSUED_BY]->(Agency)
(Organization)-[:SUBJECT_TO]->(Regulation)
(Port)-[:CONNECTED_TO]->(Port)   // trade lanes
```

모든 랩 노드에는 **`KGApplicationsLab`** 레이블도 있어 안전하게 삭제하고 재구축할 수 있습니다.


### Step 2.1a — 이전 랩 데이터 삭제

노트북을 다시 실행해도 노드가 중복되지 않아야 합니다.


In [10]:
# Step 2.1a — Clear prior lab subgraph
run_cypher(f'MATCH (n:{LAB_LABEL}) DETACH DELETE n')
print('이전 KGApplicationsLab 데이터 삭제 완료.')


이전 KGApplicationsLab 데이터 삭제 완료.


### Step 2.1b — 국가 및 수로(waterway) 생성

다른 노드가 연결할 **참조 엔티티**부터 시작합니다.


In [11]:
# Step 2.1b — Countries and waterways
run_cypher(
    f'''
    CREATE (meta:{LAB_LABEL} {{course: 'Practical KG Applications', module: 'Module_8'}})
    CREATE (nl:Country:{LAB_LABEL} {{id: 'Netherlands', name: 'Netherlands'}})
    CREATE (sg:Country:{LAB_LABEL} {{id: 'Singapore', name: 'Singapore'}})
    CREATE (pa:Country:{LAB_LABEL} {{id: 'Panama', name: 'Panama'}})
    CREATE (eg:Country:{LAB_LABEL} {{id: 'Egypt', name: 'Egypt'}})
    CREATE (dk:Country:{LAB_LABEL} {{id: 'Denmark', name: 'Denmark'}})
    CREATE (panama:Waterway:{LAB_LABEL} {{id: 'Panama_Canal', name: 'Panama Canal', type: 'canal'}})
    CREATE (suez:Waterway:{LAB_LABEL} {{id: 'Suez_Canal', name: 'Suez Canal', type: 'canal'}})
    CREATE (panama)-[:LOCATED_IN]->(pa)
    CREATE (suez)-[:LOCATED_IN]->(eg)
    '''
)
print('국가 및 수로 생성 완료.')


국가 및 수로 생성 완료.


### Step 2.1c — 항구 생성

항구는 네트워크의 **허브**입니다. Part 9 분석을 위해 처리량(throughput) 순위를 저장합니다.


In [12]:
# Step 2.1c — Major container ports
extra_countries = [
    ('China', 'China'),
    ('United States', 'United States'),
    ('South Korea', 'South Korea'),
]
for cid, cname in extra_countries:
    run_cypher(
        f'MERGE (c:Country:{LAB_LABEL} {{id: $id}}) SET c.name = $name',
        {'id': cid, 'name': cname},
    )
ports = [
    ('Port_of_Rotterdam', 'Port of Rotterdam', 'Netherlands', 1),
    ('Port_of_Singapore', 'Port of Singapore', 'Singapore', 2),
    ('Port_of_Shanghai', 'Port of Shanghai', 'China', 3),
    ('Port_of_Busan', 'Port of Busan', 'South Korea', 4),
    ('Port_of_Los_Angeles', 'Port of Los Angeles', 'United States', 5),
]
for pid, name, country, rank in ports:
    run_cypher(
        f'''
        MATCH (c:Country:{LAB_LABEL} {{id: $country}})
        CREATE (p:Port:{LAB_LABEL} {{id: $id, name: $name, throughput_rank: $rank}})
        CREATE (p)-[:LOCATED_IN]->(c)
        ''',
        {'id': pid, 'name': name, 'country': country, 'rank': rank},
    )
print(f'{len(ports)}개 항구 생성 완료.')


5개 항구 생성 완료.


### Step 2.1d — 해운 조직 생성

운송사는 항구에서 **운영**하고 전략적 수로를 **사용**합니다.


In [13]:
# Step 2.1d — Shipping organizations
run_cypher(
    f'''
    MERGE (dk:Country:{LAB_LABEL} {{id: 'Denmark'}}) SET dk.name = 'Denmark'
    MERGE (ch:Country:{LAB_LABEL} {{id: 'Switzerland'}}) SET ch.name = 'Switzerland'
    CREATE (maersk:Organization:{LAB_LABEL} {{id: 'Maersk', name: 'Maersk', sector: 'shipping'}})
    CREATE (msc:Organization:{LAB_LABEL} {{id: 'MSC', name: 'Mediterranean Shipping Company', sector: 'shipping'}})
    CREATE (cosco:Organization:{LAB_LABEL} {{id: 'COSCO', name: 'COSCO Shipping', sector: 'shipping'}})
    CREATE (maersk)-[:HEADQUARTERED_IN]->(dk)
    CREATE (msc)-[:HEADQUARTERED_IN]->(ch)
    WITH maersk, msc, cosco
    MATCH (rot:Port:{LAB_LABEL} {{id: 'Port_of_Rotterdam'}}),
          (sin:Port:{LAB_LABEL} {{id: 'Port_of_Singapore'}}),
          (sha:Port:{LAB_LABEL} {{id: 'Port_of_Shanghai'}}),
          (panama:Waterway:{LAB_LABEL} {{id: 'Panama_Canal'}}),
          (suez:Waterway:{LAB_LABEL} {{id: 'Suez_Canal'}})
    CREATE (maersk)-[:OPERATES_IN]->(rot), (maersk)-[:OPERATES_IN]->(sin)
    CREATE (maersk)-[:USES_ROUTE]->(panama), (maersk)-[:USES_ROUTE]->(suez)
    CREATE (msc)-[:OPERATES_IN]->(rot), (msc)-[:OPERATES_IN]->(sha)
    CREATE (msc)-[:USES_ROUTE]->(suez)
    CREATE (cosco)-[:OPERATES_IN]->(sha), (cosco)-[:OPERATES_IN]->(sin)
    CREATE (cosco)-[:USES_ROUTE]->(panama)
    '''
)
print('조직 및 관계 생성 완료.')


조직 및 관계 생성 완료.


### Step 2.1e — 항구 간 무역 경로(trade lane)

`CONNECTED_TO` 엣지는 **빈번한 무역 경로**를 나타냅니다(단순화를 위해 무방향).


In [14]:
# Step 2.1e — Port-to-port trade lanes
lanes = [
    ('Port_of_Rotterdam', 'Port_of_Singapore', 'Asia-Europe'),
    ('Port_of_Singapore', 'Port_of_Shanghai', 'Intra-Asia'),
    ('Port_of_Shanghai', 'Port_of_Los_Angeles', 'Trans-Pacific'),
    ('Port_of_Busan', 'Port_of_Los_Angeles', 'Trans-Pacific'),
    ('Port_of_Rotterdam', 'Port_of_Los_Angeles', 'Europe-Americas'),
]
for a, b, lane in lanes:
    run_cypher(
        f'''
        MATCH (p1:Port:{LAB_LABEL} {{id: $a}}), (p2:Port:{LAB_LABEL} {{id: $b}})
        MERGE (p1)-[:CONNECTED_TO {{lane: $lane}}]-(p2)
        ''',
        {'a': a, 'b': b, 'lane': lane},
    )
print(f'{len(lanes)}개 무역 경로 생성 완료.')


5개 무역 경로 생성 완료.


### Step 2.1f — 규정 및 규정 준수 엣지

규제 지식은 전형적인 KG 사용 사례입니다: **어떤 규칙이 누구에게 적용되나?**


In [15]:
# Step 2.1f — Regulations
run_cypher(
    f'''
    MERGE (uk:Country:{LAB_LABEL} {{id: 'United_Kingdom'}}) SET uk.name = 'United Kingdom'
    CREATE (imo:Agency:{LAB_LABEL} {{id: 'IMO', name: 'International Maritime Organization'}})
    CREATE (imo)-[:HEADQUARTERED_IN]->(uk)
    CREATE (solas:Regulation:{LAB_LABEL} {{id: 'SOLAS', name: 'Safety of Life at Sea', code: 'SOLAS'}})
    CREATE (marpol:Regulation:{LAB_LABEL} {{id: 'MARPOL', name: 'Marine Pollution Prevention', code: 'MARPOL'}})
    CREATE (solas)-[:ISSUED_BY]->(imo), (marpol)-[:ISSUED_BY]->(imo)
    WITH solas, marpol
    MATCH (o:Organization:{LAB_LABEL}) WHERE o.sector = 'shipping'
    CREATE (o)-[:SUBJECT_TO]->(solas), (o)-[:SUBJECT_TO]->(marpol)
    '''
)
print('해운 조직에 규정 연결 완료.')


해운 조직에 규정 연결 완료.


### Step 2.1g — 그래프 확인

시드 후 실행하세요. Neo4j Browser를 열고 `KGApplicationsLab` 노드를 시각화하세요.


In [16]:
# Step 2.1g — Graph summary
summary = run_cypher(
    f'''
    MATCH (n:{LAB_LABEL})
    RETURN labels(n) AS labels, count(*) AS cnt
    ORDER BY cnt DESC
    '''
)
for row in summary:
    print(row)
rels = run_cypher(
    f'''
    MATCH ()-[r]->()
    WHERE all(l IN labels(startNode(r)) WHERE l = '{LAB_LABEL}' OR l <> '{LAB_LABEL}')
    AND (startNode(r):{LAB_LABEL} OR endNode(r):{LAB_LABEL})
    RETURN type(r) AS rel, count(*) AS cnt ORDER BY cnt DESC LIMIT 10
    '''
)
print('\n상위 관계 유형:')
for row in rels:
    print(' ', row)


{'labels': ['Country', 'KGApplicationsLab'], 'cnt': 10}
{'labels': ['Port', 'KGApplicationsLab'], 'cnt': 5}
{'labels': ['Organization', 'KGApplicationsLab'], 'cnt': 3}
{'labels': ['KGApplicationsLab', 'Waterway'], 'cnt': 2}
{'labels': ['KGApplicationsLab', 'Regulation'], 'cnt': 2}
{'labels': ['KGApplicationsLab'], 'cnt': 1}
{'labels': ['KGApplicationsLab', 'Agency'], 'cnt': 1}

상위 관계 유형:
  {'rel': 'LOCATED_IN', 'cnt': 7}
  {'rel': 'OPERATES_IN', 'cnt': 6}
  {'rel': 'SUBJECT_TO', 'cnt': 6}
  {'rel': 'CONNECTED_TO', 'cnt': 5}
  {'rel': 'USES_ROUTE', 'cnt': 4}
  {'rel': 'HEADQUARTERED_IN', 'cnt': 3}
  {'rel': 'ISSUED_BY', 'cnt': 2}


---

# Part 3 — 응용: 네트워크 탐색 및 검색

**비즈니스 질문:** *유럽 항구에서 운영하는 해운 회사는 어디인가?*

SQL은 종종 많은 조인이 필요합니다. 그래프에서는 `Organization`에서 `Port`, `Country`로 **엣지를 따라갑니다**.


### Step 3.1 — 네덜란드에서 운영하는 운송사


In [17]:
# Step 3.1 — Organizations at Dutch ports
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})-[:LOCATED_IN]->(c:Country:{LAB_LABEL} {{id: 'Netherlands'}})
    RETURN o.name AS carrier, p.name AS port, c.name AS country
    ORDER BY carrier
    '''
)
for r in rows:
    print(f"{r['carrier']} → {r['port']} ({r['country']})")


Maersk → Port of Rotterdam (Netherlands)
Mediterranean Shipping Company → Port of Rotterdam (Netherlands)


### Step 3.2 — 특정 운송사가 서비스하는 모든 항구

**비즈니스 질문:** *Maersk는 어디서 운영하나?*


In [18]:
# Step 3.2 — Ports for Maersk
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL} {{id: 'Maersk'}})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})
    RETURN p.name AS port, p.throughput_rank AS global_rank
    ORDER BY global_rank
    '''
)
for r in rows:
    print(f"{r['port']} (순위 {r['global_rank']})")


Port of Rotterdam (순위 1)
Port of Singapore (순위 2)


### Step 3.3 — 매개변수화된 검색 함수

Cypher를 Python으로 감싸 제품 팀이 API에서 하나의 함수를 호출할 수 있게 합니다.


In [19]:
# Step 3.3 — Reusable discovery helper
def carriers_in_country(country_id: str) -> list[str]:
    rows = run_cypher(
        f'''
        MATCH (o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(:Port:{LAB_LABEL})-[:LOCATED_IN]->(c:Country:{LAB_LABEL} {{id: $country}})
        RETURN DISTINCT o.name AS carrier ORDER BY carrier
        ''',
        {'country': country_id},
    )
    return [r['carrier'] for r in rows]

print('네덜란드:', carriers_in_country('Netherlands'))
print('중국:', carriers_in_country('China'))


네덜란드: ['Maersk', 'Mediterranean Shipping Company']
중국: ['COSCO Shipping', 'Mediterranean Shipping Company']


---

# Part 4 — 응용: 경로 및 영향 분석

**비즈니스 질문:** *수에즈 운하(Suez Canal)가 중단되면 어떤 운송사와 항구가 영향을 받나?*

그래프는 **의존성 탐색**을 명시적으로 만듭니다: 운송사 → `USES_ROUTE` → 수로.


### Step 4.1 — 수에즈 운하를 사용하는 운송사


In [20]:
# Step 4.1 — Suez-dependent carriers
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL})-[:USES_ROUTE]->(w:Waterway:{LAB_LABEL} {{id: 'Suez_Canal'}})
    RETURN o.name AS carrier, w.name AS waterway
    ORDER BY carrier
    '''
)
for r in rows:
    print(f"{r['carrier']} — {r['waterway']} 의존")


Maersk — Suez Canal 의존
Mediterranean Shipping Company — Suez Canal 의존


### Step 4.2 — 영향받는 운송사의 하류 항구

2홉 패턴: **수로 → 운송사 → 항구**.


In [21]:
# Step 4.2 — Ports served by Suez-dependent carriers
rows = run_cypher(
    f'''
    MATCH (w:Waterway:{LAB_LABEL} {{id: 'Suez_Canal'}})<-[:USES_ROUTE]-(o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})
    RETURN DISTINCT w.name AS disrupted_route, o.name AS carrier, p.name AS affected_port
    ORDER BY carrier, affected_port
    '''
)
for r in rows:
    print(f"{r['disrupted_route']}: {r['carrier']} → {r['affected_port']}")


Suez Canal: Maersk → Port of Rotterdam
Suez Canal: Maersk → Port of Singapore
Suez Canal: Mediterranean Shipping Company → Port of Rotterdam
Suez Canal: Mediterranean Shipping Company → Port of Shanghai


### Step 4.3 — 두 항구 간 최단 무역 경로

**비즈니스 질문:** *우리 경로 모델에서 로테르담과 로스앤젤레스는 어떻게 연결되나?*


In [22]:
# Step 4.3 — Shortest path (trade lanes)
rows = run_cypher(
    f'''
    MATCH path = shortestPath(
      (a:Port:{LAB_LABEL} {{id: 'Port_of_Rotterdam'}})-[:CONNECTED_TO*]-(b:Port:{LAB_LABEL} {{id: 'Port_of_Los_Angeles'}})
    )
    RETURN [n IN nodes(path) | n.name] AS route, length(path) AS hops
    '''
)
for r in rows:
    print('경로:', ' → '.join(r['route']), f"({r['hops']} 홉)")


경로: Port of Rotterdam → Port of Los Angeles (1 홉)


### Step 4.4 — 영향 보고서 헬퍼

운영 대시보드를 위한 그래프 결과를 패키징합니다.


In [23]:
# Step 4.4 — Impact analysis function
def impact_if_waterway_blocked(waterway_id: str) -> dict:
    carriers = run_cypher(
        f'''
        MATCH (w:Waterway:{LAB_LABEL} {{id: $wid}})<-[:USES_ROUTE]-(o:Organization:{LAB_LABEL})
        RETURN collect(DISTINCT o.name) AS carriers
        ''',
        {'wid': waterway_id},
    )[0]['carriers']
    ports = run_cypher(
        f'''
        MATCH (w:Waterway:{LAB_LABEL} {{id: $wid}})<-[:USES_ROUTE]-(o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})
        RETURN collect(DISTINCT p.name) AS ports
        ''',
        {'wid': waterway_id},
    )[0]['ports']
    return {'waterway': waterway_id, 'carriers': carriers, 'ports': ports}

import json
print(json.dumps(impact_if_waterway_blocked('Suez_Canal'), indent=2))


{
  "waterway": "Suez_Canal",
  "carriers": [
    "Maersk",
    "Mediterranean Shipping Company"
  ],
  "ports": [
    "Port of Rotterdam",
    "Port of Singapore",
    "Port of Shanghai"
  ]
}


---

# Part 5 — 응용: 엔티티 해소(Entity Resolution)

**비즈니스 질문:** *"MSC"와 "Mediterranean Shipping Company"는 같은 조직인가?*

실제 데이터는 **여러 소스**에서 서로 다른 이름으로 도착합니다. 그래프는 다음을 지원합니다:

1. **후보 생성** — 유사한 이름 찾기
2. **병합** — 하나의 정규(canonical) 노드로 통합
3. **출처 추적(provenance)** — 감사를 위해 `alt_names` 유지


### Step 5.1 — 중복 레코드 시드 (두 번째 데이터 피드 시뮬레이션)


In [24]:
# Step 5.1 — Duplicate organization from another source
run_cypher(
    f'''
    CREATE (dup:Organization:{LAB_LABEL} {{id: 'MSC_DUP', name: 'MSC', alt_name: 'Mediterranean Shipping Co.'}})
    WITH dup
    MATCH (rot:Port:{LAB_LABEL} {{id: 'Port_of_Rotterdam'}})
    CREATE (dup)-[:OPERATES_IN]->(rot)
    '''
)
print('중복 MSC 레코드 생성 완료 (id=MSC_DUP).')


중복 MSC 레코드 생성 완료 (id=MSC_DUP).


### Step 5.2 — 해소 후보 찾기

간단한 규칙: 동일한 **정규화된 이름** 또는 겹치는 **항구 운영**.


In [25]:
# Step 5.2 — Candidate pairs
rows = run_cypher(
    f'''
    MATCH (a:Organization:{LAB_LABEL}), (b:Organization:{LAB_LABEL})
    WHERE id(a) < id(b) AND a.name = b.name
    RETURN a.id AS id_a, b.id AS id_b, a.name AS shared_name
    '''
)
for r in rows:
    print(f"병합 후보: {r['id_a']} + {r['id_b']} (이름={r['shared_name']})")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=3, column=11, offset=92>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 92, 'line': 3, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (a:Organization:KGApplicationsLab), (b:Organization:KGApplicationsLab)\n    WHERE id(a) < id(b) AND a.name = b.name\n    RETURN a.id AS id_a, b.id AS id_b, a.name AS shared_name\n    '
Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id i

### Step 5.3 — 중복을 정규 노드에 병합

교육 명확성을 위해 `MERGE` + `apoc.refactor.mergeNodes` 패턴을 `MATCH`로 수동 구현합니다.


In [26]:
# Step 5.3 — Merge MSC_DUP into canonical MSC
run_cypher(
    f'''
    MATCH (canonical:Organization:{LAB_LABEL} {{id: 'MSC'}})
    MATCH (dup:Organization:{LAB_LABEL} {{id: 'MSC_DUP'}})
    SET canonical.alt_names = coalesce(canonical.alt_names, []) + coalesce([dup.alt_name], [])
    WITH canonical, dup
    MATCH (dup)-[r:OPERATES_IN]->(p:Port:{LAB_LABEL})
    MERGE (canonical)-[:OPERATES_IN]->(p)
    DELETE r
    WITH canonical, dup
    DETACH DELETE dup
    '''
)
print('MSC_DUP를 MSC에 병합 완료.')


MSC_DUP를 MSC에 병합 완료.


In [27]:
# Step 5.3b — Verify single MSC node
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL}) WHERE o.name CONTAINS 'Mediterranean' OR o.id = 'MSC'
    RETURN o.id, o.name, o.alt_names
    '''
)
for r in rows:
    print(r)


{'o.id': 'MSC', 'o.name': 'Mediterranean Shipping Company', 'o.alt_names': ['Mediterranean Shipping Co.']}


---

# Part 6 — 응용: 추천 및 유사도

**비즈니스 질문:** *확장 계획을 위해 싱가포르와 유사한 항구는 어디인가?*

**협업 필터링 패턴:** **같은 운송사**에 연결된 항구는 유사합니다.
이는 *X를 산 사용자가 Y도 구매*와 같은 아이디어이지만 그래프 위에서입니다.


### Step 6.1 — 싱가포르와 운송사를 공유하는 항구


In [28]:
# Step 6.1 — Similar ports (shared carriers)
rows = run_cypher(
    f'''
    MATCH (target:Port:{LAB_LABEL} {{id: 'Port_of_Singapore'}})<-[:OPERATES_IN]-(o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(other:Port:{LAB_LABEL})
    WHERE target <> other
    RETURN other.name AS similar_port, count(DISTINCT o) AS shared_carriers
    ORDER BY shared_carriers DESC
    '''
)
for r in rows:
    print(f"{r['similar_port']}: 공유 운송사 {r['shared_carriers']}개")


Port of Rotterdam: 공유 운송사 1개
Port of Shanghai: 공유 운송사 1개


### Step 6.2 — 신규 항구에 운송사 추천 (공동 위치 패턴)

신규 항구가 **유럽**에 있다면, 이미 유럽 항구에서 운영 중인 운송사를 추천합니다.


In [29]:
# Step 6.2 — Carrier recommendations for European expansion
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})-[:LOCATED_IN]->(c:Country:{LAB_LABEL})
    WHERE c.id IN ['Netherlands', 'Denmark']
    RETURN o.name AS recommended_carrier, count(DISTINCT p) AS european_ports
    ORDER BY european_ports DESC
    '''
)
for r in rows:
    print(f"{r['recommended_carrier']}: EU 항구 {r['european_ports']}개")


Maersk: EU 항구 1개
Mediterranean Shipping Company: EU 항구 1개


---

# Part 7 — 응용: 규정 준수 및 규제 매핑

**비즈니스 질문:** *Maersk에 적용되는 IMO 규정은 무엇인가? 누가 발행했나?*

규정 준수 그래프는 **정책 → 당국 → 엔티티**를 연결합니다. 감사인은 PDF를 개별적으로 읽는 대신 엣지를 탐색합니다.


### Step 7.1 — 운송사에 적용되는 규정


In [30]:
# Step 7.1 — Maersk compliance view
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL} {{id: 'Maersk'}})-[:SUBJECT_TO]->(reg:Regulation:{LAB_LABEL})-[:ISSUED_BY]->(agency:Agency:{LAB_LABEL})
    RETURN o.name AS organization, reg.code AS regulation, agency.name AS issued_by
    ORDER BY regulation
    '''
)
for r in rows:
    print(f"{r['organization']} — {r['regulation']} 준수 필요 (발행: {r['issued_by']})")


Maersk — MARPOL 준수 필요 (발행: International Maritime Organization)
Maersk — SOLAS 준수 필요 (발행: International Maritime Organization)


### Step 7.2 — 특정 규정 하의 모든 운송사


In [31]:
# Step 7.2 — Organizations subject to SOLAS
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL})-[:SUBJECT_TO]->(reg:Regulation:{LAB_LABEL} {{code: 'SOLAS'}})
    RETURN o.name AS carrier ORDER BY carrier
    '''
)
print('SOLAS 적용 대상:', [r['carrier'] for r in rows])


SOLAS 적용 대상: ['COSCO Shipping', 'Maersk', 'Mediterranean Shipping Company']


### Step 7.3 — 규정 준수 체크리스트보내기


In [32]:
# Step 7.3 — Build compliance checklist for all shipping orgs
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL} {{sector: 'shipping'}})-[:SUBJECT_TO]->(reg:Regulation:{LAB_LABEL})
    RETURN o.name AS carrier, collect(reg.code) AS regulations
    ORDER BY carrier
    '''
)
for r in rows:
    print(f"{r['carrier']}: {', '.join(r['regulations'])}")


COSCO Shipping: SOLAS, MARPOL
Maersk: SOLAS, MARPOL
Mediterranean Shipping Company: SOLAS, MARPOL


---

# Part 8 — 응용: LLM + Cypher 비즈니스 Q&A

**비즈니스 질문:** *이해관계자는 일반 영어로 질문하고, 분석가는 모든 질문에 Cypher를 작성하지 않아야 한다.*

LangChain의 **`GraphCypherQAChain`**과 스키마 인트로스펙션을 사용합니다 — `Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`와 동일한 패턴이며 랩 그래프에 범위를 한정합니다.

> **필요:** `LLM_MODEL_SETUP.md` 및 Step 0d의 동작하는 `llm`.


### Step 8.1 — LangChain Neo4jGraph 연결


In [33]:
# Step 8.1 — LangChain Neo4jGraph (schema-aware)
from langchain_neo4j import Neo4jGraph

neo4j_graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
neo4j_graph.refresh_schema()
print('스키마 일부 (처음 800자):')
print((neo4j_graph.schema or '')[:800])


스키마 일부 (처음 800자):
Node properties:
CourseKG {module: STRING, lab: STRING, updatedAt: DATE_TIME}
Port {country: STRING, id: STRING, name: STRING, throughput_rank: INTEGER}
Organization {id: STRING, country: STRING, name: STRING, sector: STRING, alt_names: LIST}
Canal {id: STRING, country: STRING, name: STRING}
Country {id: STRING, name: STRING}
Document {lab: STRING, id: STRING, source: STRING, course: STRING, text: STRING, article_index: INTEGER, title: STRING}
Time Period {id: STRING}
Date System {id: STRING}
Event {id: STRING}
Person {id: STRING}
Concept {id: STRING}
Political Party {id: STRING}
Amendment {id: STRING}
Right {id: STRING}
Action {id: STRING}
Device {id: STRING}
Group {id: STRING}
Location {id: STRING}
LangChainLab {module: STRING, course: STRING, country: STRING, id: STRING, name: STRING, t


### Step 8.2 — GraphCypherQAChain

체인: 질문 → LLM이 Cypher 작성 → 실행 → LLM이 답변 요약.


In [34]:
# Step 8.2 — Natural language → Cypher → answer
from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain

if llm is None:
    print('건너뜀: .env에서 LLM_BACKEND를 구성하세요 (LLM_MODEL_SETUP.md 참고).')
else:
    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=neo4j_graph,
        verbose=True,
        allow_dangerous_requests=True,
        top_k=10,
    )
    question = 'Which organizations use the Suez Canal and which ports do they operate?'
    answer = chain.invoke({'query': question})
    print('\n질문:', question)
    print('답변:', answer.get('result', answer))




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Organization)-[:USES_ROUTE]->(c:Canal {name: "Suez Canal"})-[:LOCATED_IN]->(co:Country) 
WITH o, c, co
OPTIONAL MATCH (p:Port)-[:OPERATED_BY]->(o)
RETURN DISTINCT o.name AS organizationName, p.id AS portId
Full Context:
[]

> Finished chain.

질문: Which organizations use the Suez Canal and which ports do they operate?
답변: I don't know the answer.


### Step 8.3 — 직접 비즈니스 질문 시도

이 랩 그래프에서 잘 동작하는 예시:

- *Which ports are in the Netherlands?*
- *What regulations apply to COSCO?*
- *Which waterway does Maersk use?*


In [35]:
# Step 8.3 — Your question (edit the string)
MY_QUESTION = 'Which ports are connected to Rotterdam via trade lanes?'

if llm is not None:
    result = chain.invoke({'query': MY_QUESTION})
    print('질문:', MY_QUESTION)
    print('답변:', result.get('result', result))
else:
    print('이 셀을 실행하려면 LLM을 구성하세요.')




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p1:Port {name: 'Rotterdam'})-[:CONNECTED_TO*]->(p2:Port) RETURN p2.name AS port_name
Full Context:
[]

> Finished chain.
질문: Which ports are connected to Rotterdam via trade lanes?
답변: I don't know the answer.


---

# Part 9 — 응용: 그래프 분석 (허브 중요도)

**비즈니스 질문:** *무역 경로 네트워크에서 가장 연결이 많은 허브 항구는 어디인가?*

중심성(centrality) 지표는 **병목**과 **전략적 위치**를 강조합니다.
이 코스에서는 순수 Cypher로 차수 중심성(degree centrality)을 사용합니다(GDS 플러그인 불필요).


### Step 9.1 — 항구 차수 (무역 경로 이웃 수)


In [36]:
# Step 9.1 — Port connectivity rank
rows = run_cypher(
    f'''
    MATCH (p:Port:{LAB_LABEL})
    OPTIONAL MATCH (p)-[:CONNECTED_TO]-(neighbor:Port:{LAB_LABEL})
    RETURN p.name AS port, count(DISTINCT neighbor) AS lane_degree, p.throughput_rank AS throughput_rank
    ORDER BY lane_degree DESC, throughput_rank
    '''
)
for r in rows:
    print(f"{r['port']}: 이웃 {r['lane_degree']}개 (처리량 순위 {r['throughput_rank']})")


Port of Los Angeles: 이웃 3개 (처리량 순위 5)
Port of Rotterdam: 이웃 2개 (처리량 순위 1)
Port of Singapore: 이웃 2개 (처리량 순위 2)
Port of Shanghai: 이웃 2개 (처리량 순위 3)
Port of Busan: 이웃 1개 (처리량 순위 4)


### Step 9.2 — 운송사 발자국 (운영 차수)


In [37]:
# Step 9.2 — Organizations by number of ports served
rows = run_cypher(
    f'''
    MATCH (o:Organization:{LAB_LABEL})-[:OPERATES_IN]->(p:Port:{LAB_LABEL})
    RETURN o.name AS carrier, count(p) AS ports_served
    ORDER BY ports_served DESC
    '''
)
for r in rows:
    print(f"{r['carrier']}: 항구 {r['ports_served']}개")


Maersk: 항구 2개
Mediterranean Shipping Company: 항구 2개
COSCO Shipping: 항구 2개


### Step 9.3 — 선택: Neo4j Graph Data Science (GDS)

프로덕션 분석의 경우 Neo4j Desktop 또는 AuraDS에 **Graph Data Science** 라이브러리를 설치하세요.
**PageRank**, **Louvain 커뮤니티 탐지**, **betweenness centrality** 같은 알고리즘은
투영 그래프에서 실행됩니다. 9.1–9.2의 Cypher 패턴은 추가 플러그인 없이 동일한 직관을 가르칩니다.


---

# Part 10 — 마무리 및 다음 단계

## 연습한 내용

| 파트 | 응용 | 핵심 Cypher 아이디어 |
|------|-------------|-----------------|
| 3 | 네트워크 탐색 | 다중 홉 `MATCH` |
| 4 | 영향 분석 | 경로 쿼리, `shortestPath` |
| 5 | 엔티티 해소 | `MERGE`, 노드 중복 제거 |
| 6 | 추천 | 공유 이웃 패턴 |
| 7 | 규정 준수 | 정책 → 엔티티 탐색 |
| 8 | 비즈니스 Q&A | LLM + `GraphCypherQAChain` |
| 9 | 분석 | 차수 / 순위 |

## 자신의 도메인에 매핑

`Port` / `Organization`을 **자신의** 산업 엔티티로 교체하세요:

| 도메인 | 노드 예시 | 관계 예시 |
|--------|---------------|----------------------|
| 헬스케어 | Patient, Treatment, Diagnosis | `RECEIVED`, `INDICATED_FOR` |
| 금융 | Account, Transaction, Company | `TRANSFERRED_TO`, `OWNS` |
| 사이버보안 | IP, Domain, Malware | `RESOLVES_TO`, `COMMUNICATES_WITH` |
| HR | Employee, Skill, Project | `HAS_SKILL`, `WORKED_ON` |

## 학습 계속하기

1. **`Module_8_Building_Knowledge_Graphs_with_LLMs.ipynb`** — 문서에서 그래프 채우기.
2. **`Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`** — RAG, GraphRAG, agent.
3. **`Module_8_Evaluating_GraphRAG_with_RAGAS.ipynb`** — 답변 품질 측정.

## 정리 (선택)

```cypher
MATCH (n:KGApplicationsLab) DETACH DELETE n;
```

---

*코스 종료 — 실무 지식 그래프 응용*
